# HMIE Batch Validation

Validate HMIE/Scale datasets using `datamaite`. Point this notebook at a
directory containing one or more batches and it will check:

- **Scale spec compliance** -- annotation JSON conforms to the **Scale Video Playback** schema based on documentation
- **Folder structure** -- snippet directories with `seq_*/` video containers
- **Annotation coverage** -- every annotation pairs to a video (and vice-versa)
- **Video integrity** -- each video opens and decodes representative frames (optional)


**NOTE:** To run this notebook, you must first have a HMIE dataset. You can also download a mock dataset from https://gitlab.jatic.net/jatic/orchestration-interoperability/datamaite-example-datasets/-/tree/main/datasets. 

In [ ]:
import os
from pathlib import Path

import datamaite

# major.minor.patch only, so the docs don't show a hatch-vcs dev suffix in CI
print(f"datamaite {'.'.join(datamaite.__version__.split('.')[:3])}")

# -- Configuration --------------------------------------------------------

# Resolve the example datasets. CI sets DATAMAITE_DATASETS_ROOT to the cloned
# `datasets/` directory; locally it falls back to a clone beside this notebook.
# Replace the fallback with your own dataset path if needed.
DATASETS_ROOT = Path(os.environ.get("DATAMAITE_DATASETS_ROOT", "datamaite-example-datasets/datasets"))
DATA_ROOT = DATASETS_ROOT / "hmie"
BATCH_FOLDERS = (
    DATA_ROOT.joinpath("bad-corrupt-video"),
    DATA_ROOT.joinpath("bad-invalid-json"),
    DATA_ROOT.joinpath("bad-missing-task-id"),
    DATA_ROOT.joinpath("bad-orphan-annotation"),
    DATA_ROOT.joinpath("valid"),
    DATA_ROOT.joinpath("warn-fps-mismatch"),
    DATA_ROOT.joinpath("warn-orphan-video"),
)

# CHECK_VIDEO controls the per-file video probe (opening each .mp4 with
# OpenCV and decoding sample frames to detect corruption, dropped frames,
# wrong FPS, etc.).
CHECK_VIDEO = True

## Validate all the batch datasets

`validate_batches` then runs the full validation pipeline on each discovered
dataset and yields `(dataset_path, ValidationResult)` pairs as it goes -- so
the caller can stream results without buffering tens of thousands of datasets.

Each `ValidationResult` covers the four requirement checks:

- folder structure (snippet dirs, `seq_*/` containers)
- annotation coverage (every annotation paired to a video, no orphans)
- Scale spec compliance (annotation JSON against the Scale Video Playback schema)
- video integrity (per-file decode probe, only when `check_video_integrity=True`)

`result.summary()` prints a short pass/fail block per dataset; the result
object itself carries the full finding list, per-check counts, and label
histogram for downstream programmatic use.

In [ ]:
from datamaite import ValidationResult, validate, validate_batches

# Run validation on every batch
results: list[tuple[Path, ValidationResult]] = []

for batch, result in validate_batches(BATCH_FOLDERS, check_video_integrity=CHECK_VIDEO):
    results.append((batch, result))

In [ ]:
# take a look at the in-memory summary
print(results[0][1].summary())

We could also run the same validation on an individual dataset:

In [ ]:
single_result = validate(BATCH_FOLDERS[0])

single_result.summary()

## Generate an HTML rollup report

For a shareable artifact (review, hand-off, attaching to a ticket),
`render_html_report_multi` rolls every batch into **one** self-contained HTML
file built for a collection of datasets.

In [ ]:
from datamaite import render_html_report_multi

# One consolidated, self-contained report for the whole collection. Reports are
# written to an ignored output/ folder beside this notebook.
report_dir = Path("output")
report_dir.mkdir(parents=True, exist_ok=True)

collection_report = report_dir / f"{DATA_ROOT.name}-collection.html"
collection_report.write_text(render_html_report_multi(results, DATA_ROOT), encoding="utf-8")

passed = sum(1 for _, r in results if r.passed)
print(f"Collection report: {collection_report}")
print(f"  {passed}/{len(results)} datasets passed")

A rendered version of this consolidated report is published as the
[Example Validation Report](../example-validation-report.html){.external}.

### Alternatively: one report per dataset

If you prefer a standalone file for a each individual dataset, `render_html_report` renders any one `ValidationResult` on its own.

In [ ]:
from datamaite import render_html_report

for batch, result in results:
    name = str(batch).replace("/", "__")
    out = report_dir / f"{name}.html"
    out.write_text(render_html_report(result), encoding="utf-8")
    print(f"  {out}")

<hr>